# NFL Scouting Combine Metrics as Predictors of Quarterback Performance
### DATASCI 151 — Final Project | Spring 2026
**Group Members:** [Student ID 1], [Student ID 2], [Student ID 3], [Student ID 4]

---

## Introduction

The NFL Scouting Combine is a week-long event held each year in Indianapolis where college football players perform a series of athletic and cognitive tests in front of scouts and coaches from every NFL team. For quarterbacks — the most scrutinized position in football — these tests include the 40-yard dash (speed), vertical jump and 3-cone drill (agility and explosiveness), the 20-yard shuttle (short-area quickness), and the Wonderlic Personnel Test (a standardized cognitive assessment). Teams use these scores to inform their decisions in the NFL Draft, where players are selected in order of pick position across seven rounds. The central question motivating this project is: **do combine measurements actually predict how well a quarterback performs in real NFL games?**

Using NFL game data from 2004–2019, we calculate three career performance statistics for each quarterback: completion percentage, touchdown-to-interception ratio, and passing yards per game. We then merge these statistics with each player's combine testing results and examine the correlations between combine metrics and on-field performance. Our results show that no single combine measurement strongly predicts QB success, with the 40-yard dash and Wonderlic score showing especially weak relationships — suggesting that athletic combine testing captures only a narrow slice of what makes a quarterback effective in the NFL.

## Data Description
### Importing Libraries and Loading Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# Load the four tables used in this project
passer  = pd.read_csv('passer.csv')
combine = pd.read_csv('combine.csv')
draft   = pd.read_csv('draft.csv')
gp      = pd.read_csv('gameParticipation.csv')

for name, df in [('passer', passer), ('combine', combine),
                  ('draft', draft), ('gameParticipation', gp)]:
    print(f'{name:>20}: {len(df):>8,} rows,  {df.shape[1]} columns')

We use four of the twenty available tables. **`passer`** contains one row per pass attempt for every quarterback across the 2004–2019 NFL seasons (~397,000 rows), recording the outcome of each throw (completion, touchdown, interception, yards gained, etc.). **`combine`** contains the combine testing results for ~10,080 players who have participated in the event since the late 1980s. **`draft`** records the draft year, round, and pick number for ~12,140 drafted players. **`gameParticipation`** logs each player's appearance in each game, which we use to count the number of games each quarterback played in order to compute yards per game.

### Merging Tables

Our merge proceeds in four steps. First, we aggregate the `passer` play-by-play table into one row per player with career totals, and separately count each QB's total games played from `gameParticipation`. Second, we clean each of the three tables before merging. Third, we merge `combine_qb` and `draft_qb` on `playerId` (inner join). Fourth, we join that base with the cleaned `qb_stats` — again on `playerId` — giving us a single analysis-ready table with one row per QB.

In [ ]:
# ── Step 1: Aggregate passer table into career totals per player ────────────
# Passing yards only accrue on completions (passComp == 1)
passer['pass_yards'] = passer['passLength'] * passer['passComp']

qb_stats = passer.groupby('playerId').agg(
    attempts      = ('passAtt',    'sum'),
    completions   = ('passComp',   'sum'),
    touchdowns    = ('passTd',     'sum'),
    interceptions = ('passInt',    'sum'),
    total_yards   = ('pass_yards', 'sum')
).reset_index()

# ── Step 2: Count games played per QB from gameParticipation ────────────────
qb_games = (gp[gp['position'] == 'QB']
              .groupby('playerId')['gameId']
              .nunique()
              .reset_index(name='games_played'))

# Inner join — keeps only QBs present in both tables, avoids nulls in games_played
qb_stats = qb_stats.merge(qb_games, on='playerId', how='inner')

print(f'Unique QBs after inner merge: {len(qb_stats):,}')

### Data Cleaning

Three datasets required cleaning prior to analysis: `qb_stats`, `combine_qb`, and `draft_qb`. For `qb_stats`, we first examined the shape, data types, and unique values of each column. Rather than a left join, we used an inner join when merging the aggregated passer data with game participation counts, which automatically excluded the 538 quarterbacks who appeared in `passer` but had no matching record in `gameParticipation` — keeping only the 217 QBs with valid game counts. This also resolved a data type issue, as `games_played` came out as a clean integer with no nulls, requiring no further conversion. We then used `query()` to filter out quarterbacks with fewer than 100 career pass attempts, as their career averages would not be representative of sustained NFL performance. For the combined `combine` and `draft` table, `pd.unique()` revealed that the `round` column contained values ranging from 1 to 12, including 11 null entries. Rounds 8 through 12 correspond to older NFL drafts before the league standardized to seven rounds in 1994. A single `query("round <= 7")` removed both the out-of-range rounds and the null values simultaneously, since null values automatically fail any numeric comparison. The `round` column was then converted from float to integer using `pd.to_numeric()`. For `combine_qb`, no row-level cleaning was required. Missing values exist across all five drill columns — most notably the Wonderlic score, which 184 out of 337 quarterbacks in the merged table did not record — but these are structurally missing, meaning players simply opted out of or were not asked to complete certain drills. Rather than dropping these rows and losing valid data for the drills that were completed, missing values are handled on a per-metric basis during the analysis.

In [ ]:
# ── Examine qb_stats ────────────────────────────────────────────────────────
n_rows, n_cols = qb_stats.shape
print('# of rows:', n_rows)
print('# of columns:', n_cols)
print()
print(qb_stats.dtypes)
print()
print('Sample of games_played values:', pd.unique(qb_stats['games_played'])[:8])

In [ ]:
# ── Clean qb_stats ──────────────────────────────────────────────────────────

# Remove QBs with fewer than 100 career attempts — not enough data for meaningful averages
qb_stats = qb_stats.query('attempts >= 100')

# Compute the three performance metrics
qb_stats['comp_pct']       = (qb_stats['completions'] / qb_stats['attempts'] * 100).round(2)
qb_stats['td_int_ratio']   = (qb_stats['touchdowns']  / (qb_stats['interceptions'] + 1)).round(2)
qb_stats['yards_per_game'] = (qb_stats['total_yards']  / qb_stats['games_played']).round(2)

print('# of rows after cleaning:', len(qb_stats))
print()
print('Nulls remaining:')
print(qb_stats.isnull().sum())

In [ ]:
# ── Examine combine_qb ──────────────────────────────────────────────────────
combine_qb = combine[combine['combinePosition'] == 'QB'][[
    'playerId', 'combine40yd', 'combineVert', 'combineShuttle',
    'combine3cone', 'combineWonderlic', 'combineWeight', 'combineHeight'
]].copy()

n_rows, n_cols = combine_qb.shape
print('# of rows:', n_rows)
print('# of columns:', n_cols)
print()
print(combine_qb.dtypes)
print()
# No row-level cleaning needed — missing drill values are structurally missing
print('Missing values per column:')
print(combine_qb.isnull().sum())

In [ ]:
# ── Examine draft_qb ────────────────────────────────────────────────────────
draft_qb = draft[draft['position'] == 'QB'][['playerId', 'round', 'pick']].copy()

n_rows, n_cols = draft_qb.shape
print('# of rows:', n_rows)
print('# of columns:', n_cols)
print()
print(draft_qb.dtypes)
print()
print('Unique rounds:', pd.unique(draft_qb['round']))

In [ ]:
# ── Clean draft_qb ──────────────────────────────────────────────────────────

# query('round <= 7') removes rounds 8-12 (pre-1994 drafts) AND null rounds
# in one step — nulls automatically fail any numeric comparison
draft_qb = draft_qb.query('round <= 7')

# Convert round from float to int using pd.to_numeric
draft_qb['round'] = pd.to_numeric(draft_qb['round']).astype(int)

print('Unique rounds after cleaning:', pd.unique(draft_qb['round']))
print('Rows remaining:', len(draft_qb))
print()
print('Nulls remaining:')
print(draft_qb.isnull().sum())

In [ ]:
# ── Step 3: Inner merge combine_qb and draft_qb ─────────────────────────────
base = combine_qb.merge(draft_qb, on='playerId', how='inner')

# ── Step 4: Inner merge base with cleaned qb_stats ───────────────────────────
qb_master = base.merge(qb_stats, on='playerId', how='inner').reset_index(drop=True)

print(f'QBs in final merged dataset: {len(qb_master)}')
print(f'Columns: {list(qb_master.columns)}')

In [ ]:
# Confirm no duplicate players
assert qb_master['playerId'].is_unique, 'Duplicate players found'

# Missing value summary for key columns
key_cols = ['comp_pct', 'td_int_ratio', 'yards_per_game',
            'combine40yd', 'combineVert', 'combineShuttle',
            'combine3cone', 'combineWonderlic']
missing     = qb_master[key_cols].isnull().sum().rename('Missing Values')
missing_pct = (missing / len(qb_master) * 100).round(1).rename('% Missing')
display(pd.concat([missing, missing_pct], axis=1))

### Descriptive Statistics

The table below summarizes the three computed performance metrics and the five combine measurements we will use in our analysis. Combine metrics are shown for all QBs who have a recorded value.

In [ ]:
perf_cols    = ['comp_pct', 'td_int_ratio', 'yards_per_game']
combine_cols = ['combine40yd', 'combineVert', 'combineShuttle', 'combine3cone', 'combineWonderlic']

desc = qb_master[perf_cols + combine_cols].describe().loc[
    ['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']
].round(2)

desc.columns = ['Comp %', 'TD/INT Ratio', 'Yds/Game',
                '40-Yd Dash', 'Vert Jump', 'Shuttle', '3-Cone', 'Wonderlic']
desc.index   = ['N', 'Mean', 'Std Dev', 'Min', '25th %ile', 'Median', '75th %ile', 'Max']
display(desc)

---
## Results

### Correlation Between Combine Metrics and QB Performance

We compute the Pearson correlation coefficient between each combine metric and each of the three performance statistics, then visualize each pair as a scatter plot with a trend line. A loop iterates over all five combine metrics and all three performance metrics to produce the full grid.

In [ ]:
# ── Correlation table ───────────────────────────────────────────────────────
perf_labels    = {'comp_pct': 'Comp %', 'td_int_ratio': 'TD/INT Ratio', 'yards_per_game': 'Yds/Game'}
combine_labels = {
    'combine40yd':      '40-Yd Dash',
    'combineVert':      'Vert Jump',
    'combineShuttle':   'Shuttle',
    'combine3cone':     '3-Cone Drill',
    'combineWonderlic': 'Wonderlic',
}

corr_rows = []
for c_col, c_label in combine_labels.items():
    row = {'Combine Metric': c_label}
    for p_col, p_label in perf_labels.items():
        sub = qb_master[[c_col, p_col]].dropna()
        r   = sub[c_col].corr(sub[p_col])
        row[p_label] = round(r, 3)
        row[f'n ({p_label})'] = len(sub)
    corr_rows.append(row)

corr_df = pd.DataFrame(corr_rows).set_index('Combine Metric')
# Display only the r-value columns in a clean table
display(corr_df[list(perf_labels.values())].style
        .background_gradient(cmap='RdBu', axis=None, vmin=-0.5, vmax=0.5)
        .format('{:.3f}'))

In [ ]:
# ── Heatmap of correlations ──────────────────────────────────────────────────
heat_data = corr_df[list(perf_labels.values())]

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(heat_data, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-0.5, vmax=0.5,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Pearson r'})
ax.set_title('Correlation: Combine Metrics vs QB Performance', fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

**Reading the heatmap:** Values close to +1 (dark blue) indicate a positive relationship; values close to −1 (dark red) indicate a negative relationship; values near 0 (white) indicate no linear relationship. For speed metrics like the 40-yard dash, shuttle, and 3-cone drill, a *negative* correlation with performance would be desirable (lower time = faster athlete). The heatmap shows that no combine metric correlates strongly with any of the three QB performance stats — all values are close to zero, with the strongest signal being a slight negative relationship between agility drills (shuttle, 3-cone) and TD/INT ratio.

In [ ]:
# ── Scatter plot grid: combine metric (rows) x performance stat (cols) ──────
fig, axes = plt.subplots(
    nrows=len(combine_labels),
    ncols=len(perf_labels),
    figsize=(13, 18)
)

colors = ['#1f77b4', '#2ca02c', '#d62728']

for i, (c_col, c_label) in enumerate(combine_labels.items()):
    for j, (p_col, p_label) in enumerate(perf_labels.items()):
        ax  = axes[i][j]
        sub = qb_master[[c_col, p_col]].dropna()
        r   = sub[c_col].corr(sub[p_col])

        ax.scatter(sub[c_col], sub[p_col],
                   alpha=0.5, s=50, color=colors[j],
                   edgecolors='white', linewidths=0.3)

        # Trend line
        m, b = np.polyfit(sub[c_col], sub[p_col], 1)
        x_line = np.linspace(sub[c_col].min(), sub[c_col].max(), 200)
        ax.plot(x_line, m * x_line + b, color='black', linewidth=1.5, linestyle='--')

        # Correlation and n label
        ax.text(0.97, 0.97, f'r = {r:.3f}\nn = {len(sub)}',
                transform=ax.transAxes, ha='right', va='top', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))

        ax.set_xlabel(c_label, fontsize=10)
        ax.set_ylabel(p_label, fontsize=10)
        ax.set_title(f'{c_label} vs {p_label}', fontsize=10, fontweight='bold')

plt.suptitle('Combine Metrics vs QB Career Performance Statistics',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Interpretation:** Across all 15 scatter plots, the trend lines are nearly flat and the r-values are small, confirming that none of the five combine metrics is a meaningful predictor of any of the three QB performance statistics. A few observations stand out:

- **40-Yard Dash:** Near-zero correlation with all three metrics. Faster QBs (lower dash times) do not systematically complete more passes or throw more touchdowns than slower ones. This is unsurprising — QB success depends overwhelmingly on pocket presence and arm accuracy, not straight-line speed.
- **Vertical Jump:** Essentially uncorrelated with performance. Jumping ability has no obvious connection to passing efficiency.
- **Shuttle & 3-Cone Drill:** Small negative correlations with TD/INT ratio (r ≈ −0.16 and −0.17), suggesting that QBs who tested *faster* on short-area agility drills had marginally better TD/INT ratios — but the signal is weak and the scatter is wide.
- **Wonderlic Score:** The cognitive test that scouts have long debated shows r ≈ 0.10 with TD/INT ratio and near-zero with the other two metrics. Higher scores do not translate into meaningfully better accuracy or efficiency at the NFL level.

---
### Are Combine Scores Tied to Draft Position?

Before concluding, it is worth asking a related question: even if combine metrics do not predict on-field performance, do they at least predict *how highly a QB is drafted*? If scouts are placing significant weight on combine scores when making draft decisions, we would expect better combine performances (faster 40-yard dash, higher vertical jump, lower 3-cone time) to correlate with earlier pick numbers. We test this using two approaches: a correlation table between each combine metric and both draft round and overall pick number, and box plots showing the distribution of each combine metric across draft rounds 1–7.

In [ ]:
# ── Correlation: combine metrics vs draft round and overall pick number ──────
draft_corr_rows = []
for c_col, c_label in combine_labels.items():
    sub = qb_master[['round', 'pick', c_col]].dropna()
    r_round = sub['round'].corr(sub[c_col])
    r_pick  = sub['pick'].corr(sub[c_col])
    draft_corr_rows.append({
        'Combine Metric': c_label,
        'r vs Draft Round': round(r_round, 3),
        'r vs Overall Pick': round(r_pick, 3),
        'N': len(sub)
    })

draft_corr_df = pd.DataFrame(draft_corr_rows).set_index('Combine Metric')
display(draft_corr_df.style
        .background_gradient(cmap='RdBu_r', subset=['r vs Draft Round', 'r vs Overall Pick'],
                             axis=None, vmin=-0.5, vmax=0.5)
        .format({'r vs Draft Round': '{:.3f}', 'r vs Overall Pick': '{:.3f}', 'N': '{:d}'}))

**Reading the table:** For speed-based metrics (40-yd dash, shuttle, 3-cone), a *positive* r-value means slower athletes tend to be drafted later — the expected direction. For jump-based metrics (vertical), a *negative* r-value means higher jumpers tend to be drafted earlier. The Wonderlic score shows a near-zero correlation with both draft round and overall pick, confirming that cognitive scores play almost no role in where QBs are drafted.

In [ ]:
# ── Box plots: combine metric distributions by draft round (rounds 1-7) ─────
qb_rounds_only = qb_master[qb_master['round'].between(1, 7)].copy()

fig, axes = plt.subplots(1, len(combine_labels), figsize=(18, 5))

round_colors = ['#1a6faf','#4a9fd4','#7abfef','#f4a460','#e07b39','#c0392b','#922b21']

for ax, (c_col, c_label) in zip(axes, combine_labels.items()):
    data_by_round = [
        qb_rounds_only[qb_rounds_only['round'] == r][c_col].dropna().values
        for r in range(1, 8)
    ]
    bp = ax.boxplot(data_by_round, positions=range(1, 8), widths=0.55,
                    patch_artist=True,
                    medianprops=dict(color='white', linewidth=2),
                    whiskerprops=dict(color='gray', linewidth=1),
                    capprops=dict(color='gray', linewidth=1),
                    flierprops=dict(marker='o', markersize=3,
                                   alpha=0.4, linestyle='none'))

    for patch, color in zip(bp['boxes'], round_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)

    # Annotate sample sizes
    for r, data in zip(range(1, 8), data_by_round):
        ax.text(r, ax.get_ylim()[0] if ax.get_ylim()[0] > 0 else ax.get_ylim()[0],
                f'n={len(data)}', ha='center', va='top', fontsize=7.5, color='gray')

    sub = qb_master[['round', c_col]].dropna()
    r_val = sub[sub['round'].between(1,7)]['round'].corr(sub[sub['round'].between(1,7)][c_col])
    ax.set_xlabel('Draft Round', fontsize=11)
    ax.set_ylabel(c_label, fontsize=11)
    ax.set_title(f'{c_label}\nr = {r_val:.3f}', fontsize=11, fontweight='bold')
    ax.set_xticks(range(1, 8))

plt.suptitle('Combine Metric Distributions by Draft Round (Rounds 1–7)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter: overall pick number vs the two strongest combine predictors ─────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

top_metrics = [
    ('combine3cone',  '3-Cone Drill (sec)', axes[0], '#d62728'),
    ('combineVert',   'Vertical Jump (in)',  axes[1], '#1a6faf'),
]

for c_col, c_label, ax, color in top_metrics:
    sub = qb_master[['pick', c_col]].dropna()
    r   = sub['pick'].corr(sub[c_col])

    ax.scatter(sub[c_col], sub['pick'],
               alpha=0.5, s=55, color=color,
               edgecolors='white', linewidths=0.3)

    m, b = np.polyfit(sub[c_col], sub['pick'], 1)
    x_line = np.linspace(sub[c_col].min(), sub[c_col].max(), 200)
    ax.plot(x_line, m * x_line + b, color='black', linewidth=1.8, linestyle='--')

    ax.text(0.97, 0.97, f'r = {r:.3f}\nn = {len(sub)}',
            transform=ax.transAxes, ha='right', va='top', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.85))

    ax.set_xlabel(c_label, fontsize=12)
    ax.set_ylabel('Overall Draft Pick Number', fontsize=12)
    ax.set_title(f'{c_label} vs Overall Pick', fontsize=12, fontweight='bold')

plt.suptitle('Strongest Combine Predictors of QB Draft Position',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Interpretation:** The results show that combine scores *do* have a modest relationship with draft position — stronger than their relationship with actual on-field performance. The **3-cone drill** (r ≈ 0.30) and **vertical jump** (r ≈ −0.25) are the strongest predictors: QBs who ran a faster 3-cone drill and jumped higher tended to be drafted earlier. The **40-yard dash** shows a similar but slightly weaker pattern (r ≈ 0.24). The **Wonderlic score**, however, is almost entirely uncorrelated with draft position (r ≈ 0.02), suggesting scouts largely ignore it despite its widespread use and media attention.

Taken together with the earlier analysis, this reveals an interesting disconnect: scouts *do* reward athleticism at the combine with higher draft picks, but those same athletic metrics do not translate into better on-field QB performance. This suggests that NFL front offices may be overweighting physical combine results relative to other factors — such as college production, film study, or football IQ measures — when evaluating quarterback prospects.

---
## Discussion

This project asked whether NFL Scouting Combine metrics predict quarterback performance as measured by three career statistics: completion percentage, TD/INT ratio, and passing yards per game. Using data from 182 quarterbacks who both attended the combine and recorded at least 100 career pass attempts between 2004 and 2019, we found that no combine metric — the 40-yard dash, vertical jump, 20-yard shuttle, 3-cone drill, or Wonderlic score — correlates strongly with any of the three performance measures. Correlation coefficients ranged from −0.21 to +0.17 across all metric-performance pairs, with most clustering near zero.

These findings have a clear implication: combine testing, taken alone, is a poor predictor of quarterback effectiveness in the NFL. The skills that drive QB success — reading defenses pre-snap, going through progressions under pressure, delivering accurate throws with anticipation — are simply not captured by a stopwatch or a paper-and-pencil cognitive test. This does not mean the combine is worthless; rather, it suggests that its value lies in identifying physical *floors* (e.g., flagging players with injury concerns) rather than predicting *ceilings*. Future work could incorporate college-level performance statistics, advanced NFL metrics like Expected Points Added (EPA), or film-based evaluation scores to build a richer picture of what actually predicts QB success.